# Step 1: Load and inspect raw ETF files

We load the three Kaggle price files (SPY, TLT, XLF), convert the `Date` column to real dates using a strict format so any malformed date stops the notebook, and sort each table by date. We then check for repeated dates and record each file's first date, last date and number of rows. These checks confirm the files cover the expected period (25 Feb 2005 – 10 Nov 2017) before anything is built on them.

In [1]:
RAW = 'data/raw/' # relative to the notebook's working folder
import os; print(os.listdir(RAW)) # should list the 4 files

['spy.us.txt', 'tlt.us.txt', 'VIXCLS.csv', 'xlf.us.txt']


In [2]:
import os
import pandas as pd

FILES = {"SPY": "spy.us.txt", "TLT": "tlt.us.txt", "XLF": "xlf.us.txt"}

def load_etf(path):
    df = pd.read_csv(path)

    # strict: raises on bad dates, so pandas stops with an error instead of carrying on.
    df["Date"] = pd.to_datetime(df["Date"], format="%Y-%m-%d")
    df = df.sort_values("Date", kind="stable").reset_index(drop=True)

    return df

def summarise(ticker, df):
    dup_dates = df.loc[df["Date"].duplicated(keep=False), "Date"].unique()
    return {"ticker": ticker,
            "start": df["Date"].min().date(),
            "end": df["Date"].max().date(),
            "rows": len(df),
            "n_dup_dates": len(dup_dates)}

data = {t: load_etf(os.path.join(RAW, f)) for t, f in FILES.items()}
summary_df = pd.DataFrame([summarise(t, df) for t, df in data.items()]).set_index("ticker")
summary_df

,start,end,rows,n_dup_dates
ticker,,,,
SPY,2005-02-25,2017-11-10,3201,0
TLT,2005-02-25,2017-11-10,3201,0
XLF,2005-02-25,2017-11-10,3201,0


# Step 2: Clean and sanity-check prices

We drop the `OpenInt` column after confirming it is all zeros, since open interest has no meaning for an ETF. We then flag rows that are physically impossible: a High below the Low, a price of zero or less, or negative volume. These rows are reported, not deleted. Genuine extreme days, such as the 2010 flash crash, are kept, because they are exactly the high-volatility events we want to forecast.

In [3]:
PRICE_COLS = ["Open", "High", "Low", "Close"]

def clean_and_check(ticker, df):
    # 1. OpenInt: confirm it carries no information, then drop it
    assert (df["OpenInt"] == 0).all(), f"[{ticker}] OpenInt has non-zero values"
    df = df.drop(columns="OpenInt")

    # 2. Sanity checks: one boolean mask per rule (True = impossible row)
    checks = {
        "high_below_low": df["High"] < df["Low"],
        "price_not_positive": (df[PRICE_COLS] <= 0).any(axis=1),
        "negative_volume": df["Volume"] < 0,
    }

    # 3. Report, don't delete: print how many rows break each rule
    for rule, mask in checks.items():
        n_bad = int(mask.sum())
        print(f"[{ticker}] {rule}: {n_bad} rows")
        if n_bad:
            print("   dates:", df.loc[mask, "Date"].dt.strftime("%Y-%m-%d").tolist())

    return df

# Apply to the three tables from step 1
data = {t: clean_and_check(t, df) for t, df in data.items()}

[SPY] high_below_low: 0 rows
[SPY] price_not_positive: 0 rows
[SPY] negative_volume: 0 rows
[TLT] high_below_low: 0 rows
[TLT] price_not_positive: 0 rows
[TLT] negative_volume: 0 rows
[XLF] high_below_low: 0 rows
[XLF] price_not_positive: 0 rows
[XLF] negative_volume: 0 rows


# Step 3: Merge VIX and align to SPY trading days

We add the VIX (series VIXCLS from FRED) and combine all four files into one table with one row per SPY trading day. Where TLT, XLF or the VIX has no value for a SPY day, we carry the last known value forward, never backward, so no row ever uses information from the future. A before/after table shows the percentage of missing values in each column.

In [4]:
# 3a. Load the VIX from FRED; holidays are "." or blank, so read them as missing
vix = pd.read_csv(RAW + "VIXCLS.csv", na_values=".")
vix = vix.rename(columns={vix.columns[0]: "Date", "VIXCLS": "VIX"})  # date column name varies by download
vix["Date"] = pd.to_datetime(vix["Date"], format="%Y-%m-%d")

# 3b. Prefix TLT/XLF columns so they don't clash with SPY's (e.g. "TLT_Close")
def add_prefix(df, ticker):
    return df.rename(columns={c: f"{ticker}_{c}" for c in df.columns if c != "Date"})

tlt = add_prefix(data["TLT"], "TLT")
xlf = add_prefix(data["XLF"], "XLF")

# 3c. Start from SPY's dates and left-join the others onto them
merged = data["SPY"]
for other in [tlt, xlf, vix]:
    merged = merged.merge(other, on="Date", how="left", validate="one_to_one")
assert len(merged) == len(data["SPY"]), "Merge changed the number of SPY rows"

# 3d. % missing per column BEFORE filling
missing_before = merged.isna().mean() * 100

# 3e. Forward-fill only (never backward): copy the last known value onto gaps
fill_cols = [c for c in merged.columns if c.startswith(("TLT_", "XLF_")) or c == "VIX"]
merged[fill_cols] = merged[fill_cols].ffill()

# 3f. % missing AFTER, side by side with before
missing_after = merged.isna().mean() * 100
missing_table = pd.DataFrame({"before_%": missing_before, "after_%": missing_after}).round(2)
missing_table

,before_%,after_%
Date,0.0,0.0
Open,0.0,0.0
High,0.0,0.0
Low,0.0,0.0
Close,0.0,0.0
Volume,0.0,0.0
TLT_Open,0.0,0.0
TLT_High,0.0,0.0
TLT_Low,0.0,0.0
TLT_Close,0.0,0.0


# Step 4 — Compute returns, target and memory features (1–3)

We compute SPY's daily log returns and, from them, realised volatility (RV) over 1, 5 and 22 trading days: yesterday's shock, last week and last month. These three features form the HAR-RV benchmark. We also build the target: the log of realised volatility over the *next* 5 trading days. On 11 days (10 in training, 1 in test) SPY's close repeats exactly, giving a zero return whose log is −∞. We raise these to a floor equal to the smallest non-zero absolute return in the training period, computed on training data only, so no test information is used and no genuine trading day is altered. A final check confirms that only the zero-return days were changed.

In [5]:
import numpy as np

TRAIN_END = "2014-12-31"   # last training day (plan: train 2005-2014)

def log_returns(close):
    """Daily log return: r_t = ln(Close_t / Close_{t-1}). First row is NaN."""
    return np.log(close).diff()

def log_rv(r, n, floor):
    """Log realised volatility over the n days UP TO AND INCLUDING t (usable as a feature).
    RV = sqrt(sum of squared returns); n=1 gives |r_t|. Values below `floor` are raised to it."""
    rv = np.sqrt(r.pow(2).rolling(n).sum())
    return np.log(rv.clip(lower=floor))

def log_frv(r, n, floor):
    """Log realised volatility over the n days AFTER t (target only, never a feature)."""
    return log_rv(r, n, floor).shift(-n)

# 4a. SPY daily log returns
merged["r"] = log_returns(merged["Close"])

# 4b. Zero-return check: days where the close repeats exactly (prices have only 2-3 decimals)
is_zero  = merged["r"] == 0
is_train = merged["Date"] <= TRAIN_END
print(f"Zero-return days: {is_zero.sum()} (train: {(is_zero & is_train).sum()}, test: {(is_zero & ~is_train).sum()})")
print(merged.loc[is_zero, "Date"].dt.strftime("%Y-%m-%d").tolist())

# 4c. Floor = smallest NON-ZERO |r| in the training period (training only -> no test information).
#     Zero days get the smallest move we can actually measure; no real day is changed.
FLOOR = merged.loc[is_train & ~is_zero, "r"].abs().min()
print(f"Floor: {FLOOR:.2e} (log = {np.log(FLOOR):.2f})")

# 4d. Target: log RV over the next 5 trading days
merged["y"] = log_frv(merged["r"], 5, FLOOR)

# 4e. Features 1-3 (HAR memory): yesterday's shock, last week, last month
merged["f1"] = log_rv(merged["r"], 1, FLOOR)    # log RV_1 = log|r_t|
merged["f2"] = log_rv(merged["r"], 5, FLOOR)    # log RV_5
merged["f3"] = log_rv(merged["r"], 22, FLOOR)   # log RV_22

# 4f. Sanity check: the floor changed only the zero-return days
with np.errstate(divide="ignore"):
    changed = (merged["f1"] != np.log(merged["r"].abs())) & merged["r"].notna()
assert changed.sum() == is_zero.sum(), "Floor altered non-zero days"

Zero-return days: 11 (train: 10, test: 1)
['2005-08-22', '2005-11-08', '2007-12-19', '2012-07-30', '2012-10-31', '2013-03-27', '2013-06-10', '2013-07-30', '2013-10-21', '2014-02-26', '2017-01-10']
Floor: 3.27e-05 (log = -10.33)


# Step 5 — Build the leverage feature (5)

We add SPY's 1-day log return with its sign kept (f5 = rₜ). Feature 1 already measures the size of today's move; this feature adds its direction, because falls tend to raise future volatility more than rises of the same size (the leverage effect). It uses only day t's return, so it is known at time t and leaks nothing from the future. A quick check on the training period confirms the effect in our data: after the 10% biggest falls, next-week log volatility averages −3.44, against −3.70 after the 10% biggest rises and −4.03 overall.

In [6]:
# 5a. Feature 5 (leverage effect): today's return WITH its sign.
#     f1 already holds the SIZE of today's move (log|r_t|); f5 adds its DIRECTION.
merged["f5"] = merged["r"]

# 5b. Sanity check: f5 uses only day t's return (no shift), so it is known at t -> no leakage
assert merged["f5"].equals(merged["r"]), "f5 must equal today's return"

# 5c. Leverage-effect check, on TRAINING data only (the test set stays untouched until Sat 14 Nov):
#     average next-week log volatility after the biggest falls vs the biggest rises
train = merged[merged["Date"] <= TRAIN_END]
low, high = train["f5"].quantile([0.10, 0.90])
print(f"Mean y after the 10% biggest falls: {train.loc[train['f5'] <= low,  'y'].mean():.2f}")
print(f"Mean y after the 10% biggest rises: {train.loc[train['f5'] >= high, 'y'].mean():.2f}")
print(f"Mean y on all training days:        {train['y'].mean():.2f}")

Mean y after the 10% biggest falls: -3.44
Mean y after the 10% biggest rises: -3.70
Mean y on all training days:        -4.03


# Step 6 — Build VIX and cross-asset features (4, 6–8)

We add four features that look beyond SPY's own price history. Feature 4 is the log of the VIX, the market's own forecast of volatility. Features 6 and 7 reuse the realised-volatility code from step 4 on the bond ETF (TLT) and the financials ETF (XLF): bond-market turbulence, and banking stress relative to the whole market (the log of XLF's 22-day volatility divided by SPY's). Feature 8 is TLT's 22-day return, a flight-to-safety signal. All four use data up to and including day t, aligned to SPY's trading days in step 3. A check confirms that the volatility floor never changes a 22-day value, and the summary below shows plausible ranges: the VIX runs from 9 to 81 (the 2008 peak), and XLF is typically about 1.4 times as volatile as SPY (mean f7 = 0.36).

In [7]:
# 6a. Feature 4: log of today's VIX close (known at the close of day t, so no look-ahead)
assert (merged["VIX"] > 0).all(), "VIX must be positive before taking the log"
merged["f4"] = np.log(merged["VIX"])

# 6b. TLT and XLF daily log returns, then log RV over 22 days with the same functions and floor as SPY
merged["TLT_r"] = log_returns(merged["TLT_Close"])
merged["XLF_r"] = log_returns(merged["XLF_Close"])

# 6c. Features 6-8
merged["f6"] = log_rv(merged["TLT_r"], 22, FLOOR)                     # bond-market turbulence
merged["f7"] = log_rv(merged["XLF_r"], 22, FLOOR) - merged["f3"]      # log(XLF RV_22 / SPY RV_22): banks vs market
merged["f8"] = np.log(merged["TLT_Close"]).diff(22)                   # TLT 22-day return (flight to safety)

# 6d. Sanity checks: the floor exists for single zero-return days, so it must never bind on a 22-day window
for t in ["TLT", "XLF"]:
    rv22 = np.sqrt(merged[f"{t}_r"].pow(2).rolling(22).sum())
    assert (rv22.dropna() > FLOOR).all(), f"[{t}] floor altered a 22-day RV"
print(f"VIX range: {merged['VIX'].min():.2f} to {merged['VIX'].max():.2f} (log: {merged['f4'].min():.2f} to {merged['f4'].max():.2f})")
merged[["f4", "f6", "f7", "f8"]].describe().round(3)

VIX range: 9.14 to 80.86 (log: 2.21 to 4.39)


,f4,f6,f7,f8
count,3201.000,3179.000,3179.000,3179.000
mean,2.855,-3.310,0.364,0.004
std,0.387,0.356,0.255,0.039
min,2.213,-4.475,-0.236,-0.153
25%,2.561,-3.559,0.187,-0.020
50%,2.773,-3.352,0.314,0.003
75%,3.084,-3.061,0.517,0.026
max,4.393,-2.232,1.332,0.245


# Step 7 — Build activity and trend features (9–10)

Feature 9 compares today's trading volume with a normal day this month: the log of today's volume divided by the mean volume of the last 22 days, today included. It is close to 0 on an average day and positive on unusually busy days. Feature 10 is SPY's 22-day log return, the market's trend over the last month. Both use only data up to day t. Two checks run: volume is never zero (the log would be infinite), and the 22-day return equals the sum of the last 22 daily returns.

In [8]:
# 7a. Feature 9: log(volume today / mean volume of the 22 days ending today). The window includes day t,
#     so it is known at the close of t.
assert (merged["Volume"] > 0).all(), "zero volume would make log(Volume / mean) infinite"
merged["f9"] = np.log(merged["Volume"] / merged["Volume"].rolling(22).mean())

# 7b. Feature 10: SPY 22-day log return = ln(Close_t / Close_{t-22})
merged["f10"] = np.log(merged["Close"]).diff(22)

# 7c. Sanity check: a 22-day log return is the sum of the last 22 daily log returns
assert np.allclose(merged["f10"].dropna(), merged["r"].rolling(22).sum().dropna()), "f10 != sum of 22 returns"
merged[["f9", "f10"]].describe().round(3)

,f9,f10
count,3180.000,3179.000
mean,-0.045,0.006
std,0.326,0.046
min,-1.746,-0.364
25%,-0.250,-0.012
50%,-0.055,0.013
75%,0.150,0.032
max,1.321,0.195


# Step 8 — Trim incomplete rows, split and save model table

The first 22 rows have no full 22-day window and the last 5 have no target yet (it needs the next 5 trading days), so we remove 27 rows. We label each remaining row `train` up to 31 Dec 2014 and `test` from 1 Jan 2015. We then apply the embargo: the last 5 training rows (24–31 Dec 2014) are removed, because their targets use returns up to 8 Jan 2015, which belong to the test period. A check confirms that every remaining training target ends on or before 31 Dec 2014. The saved table has 2,452 training rows (30 Mar 2005 – 23 Dec 2014) and 717 test rows (2 Jan 2015 – 3 Nov 2017), and it contains only `train` and `test` rows, so no model can pick up an embargoed row by accident. The test set ends on 3 Nov, not 10 Nov, because the last 5 days have no target.

In [9]:
FEATURES = [f"f{i}" for i in range(1, 11)]

# 8a. Incomplete rows: the first rows (22-day windows not yet full) and the last 5 (target not yet known)
complete = merged[["y"] + FEATURES].notna().all(axis=1)
first, last = complete.idxmax(), complete[::-1].idxmax()
assert complete.loc[first:last].all(), "NaN inside the table, not only at its ends"
print(f"Dropped at the start: {first} rows (up to {merged.loc[first - 1, 'Date']:%Y-%m-%d})")
print(f"Dropped at the end:   {len(merged) - 1 - last} rows (from {merged.loc[last + 1, 'Date']:%Y-%m-%d})")

Dropped at the start: 22 rows (up to 2005-03-29)
Dropped at the end:   5 rows (from 2017-11-06)


In [10]:
# 8b. Keep complete rows, rename Date -> date, and label each row train or test by date
table = merged.loc[complete, ["Date", "y"] + FEATURES].rename(columns={"Date": "date"})
table["split"] = np.where(table["date"] <= TRAIN_END, "train", "test")

# 8c. Embargo: remove the last 5 training rows. Row t's target uses the 5 trading days after t, so
#     for these rows it reaches into the first days of 2015, which belong to the test period.
window_end = merged["Date"].shift(-5)                  # last day whose return enters row t's target
embargo = table.index[table["split"] == "train"][-5:]
print("Embargoed rows (date -> last day used by its target):")
print(pd.DataFrame({"date": table.loc[embargo, "date"], "target_window_ends": window_end[embargo]}).to_string(index=False))
assert (window_end[embargo] > TRAIN_END).all(), "embargo removed rows that did not need removing"
table = table.drop(index=embargo)

# 8d. Leakage check on what is kept: no training target may reach past the end of the training period
kept_train = table.index[table["split"] == "train"]
assert (window_end[kept_train] <= TRAIN_END).all(), "a training target reaches into the test period"
table = table.reset_index(drop=True)

Embargoed rows (date -> last day used by its target):
      date target_window_ends
2014-12-24         2015-01-02
2014-12-26         2015-01-05
2014-12-29         2015-01-06
2014-12-30         2015-01-07
2014-12-31         2015-01-08


In [11]:
# 8e. Final checks, then save. Raw files are untouched; this is the only table the models load.
assert table["date"].is_monotonic_increasing and table["date"].is_unique
assert table[["y"] + FEATURES].notna().all().all() and np.isfinite(table[["y"] + FEATURES]).all().all()

os.makedirs("data/processed", exist_ok=True)
table.to_csv("data/processed/model_table.csv", index=False, date_format="%Y-%m-%d")

by_split = table.groupby("split")["date"].agg(rows="size", first="min", last="max")
by_split["first"], by_split["last"] = by_split["first"].dt.date, by_split["last"].dt.date
by_split

,rows,first,last
split,,,
test,717,2015-01-02,2017-11-03
train,2452,2005-03-30,2014-12-23


# Step 9 — Set up CV splitter, metrics and prediction export

The shared tools live in `src/foundation/`, so every slice imports the same code: `make_cv()` (scikit-learn `TimeSeriesSplit` with 5 splits and a 5-row gap, used on training rows only), `evaluate()` and `save_predictions()`. `evaluate()` returns MAE, RMSE, out-of-sample R² (measured against the mean of the scored sample) and QLIKE, all on the log scale; QLIKE compares variances and punishes under-predicting volatility more than over-predicting it. `save_predictions()` writes `results/preds_<model>.csv` and refuses a file that does not cover exactly the test dates, so every model is compared on the same days. The cells below show the five CV folds, check `evaluate()` on toy numbers, give a first CV score for the naive baseline (forecast = last week's volatility), and test the export in a temporary folder. The test set is not scored here.

In [12]:
import tempfile
from src.foundation.split import load_model_table, make_cv      # run from the repo root so `src` is found
from src.foundation.evaluate import evaluate, save_predictions

# 9a. Reload the saved table exactly as every slice will, and show the CV folds (training rows only)
mt = load_model_table()
train = mt[mt["split"] == "train"].reset_index(drop=True)
folds = []
for k, (fit, val) in enumerate(make_cv().split(train), 1):
    # the target at row t uses days t+1..t+5, so the first scored row must be at least 6 rows after the last fitted one
    assert val.min() - fit.max() > 5, f"fold {k}: fitting and scoring blocks overlap"
    folds.append({"fold": k, "fit_rows": len(fit), "fit_end": train.loc[fit.max(), "date"].date(),
                  "val_rows": len(val), "val_start": train.loc[val.min(), "date"].date(),
                  "val_end": train.loc[val.max(), "date"].date()})
pd.DataFrame(folds).set_index("fold")

,fit_rows,fit_end,val_rows,val_start,val_end
fold,,,,,
1,407,2006-11-06,408,2006-11-14,2008-06-30
2,815,2008-06-23,408,2008-07-01,2010-02-11
3,1223,2010-02-04,408,2010-02-12,2011-09-26
4,1631,2011-09-19,408,2011-09-27,2013-05-13
5,2039,2013-05-06,408,2013-05-14,2014-12-23


In [13]:
# 9b. Sanity checks of evaluate() on toy numbers
y = np.linspace(-5, -3, 200)
perfect = evaluate(y, y)
assert perfect["MAE"] == perfect["RMSE"] == perfect["QLIKE"] == 0 and perfect["R2"] == 1

# Same size of miss, opposite direction: MAE and RMSE agree, QLIKE punishes the under-prediction more
low  = evaluate(y, y - 0.5)     # forecast volatility too low
high = evaluate(y, y + 0.5)     # forecast volatility too high
assert np.isclose(low["MAE"], high["MAE"]) and low["QLIKE"] > high["QLIKE"]
print(f"QLIKE after under-predicting by 0.5: {low['QLIKE']:.3f}   after over-predicting by 0.5: {high['QLIKE']:.3f}")

# 9c. First CV score: the naive baseline (forecast = last week's RV, f2) on each scoring block.
#     Training rows only; the test set is not touched.
pd.DataFrame([evaluate(train.loc[val, "y"], train.loc[val, "f2"]) for _, val in make_cv().split(train)],
             index=pd.RangeIndex(1, 6, name="fold")).round(3)

QLIKE after under-predicting by 0.5: 0.718   after over-predicting by 0.5: 0.368


,MAE,RMSE,R2,QLIKE
fold,,,,
1,0.464,0.621,-0.229,2.068
2,0.367,0.480,0.432,0.725
3,0.433,0.569,0.130,1.013
4,0.467,0.607,-0.179,1.216
5,0.444,0.552,-0.284,0.795


In [14]:
# 9d. Plumbing check for save_predictions(): write to a temporary folder (never results/) with a constant
#     forecast. Nothing is scored here; the point is that the file format is right and bad inputs are refused.
test = mt[mt["split"] == "test"]
const = np.full(len(test), train["y"].mean())
with tempfile.TemporaryDirectory() as tmp:
    path = save_predictions("demo", test["date"], test["y"], const, results_dir=tmp)
    back = pd.read_csv(path)
    assert list(back.columns) == ["date", "y_true", "y_pred"] and len(back) == len(test)
    print(back.head(3).to_string(index=False))
    try:
        save_predictions("demo", test["date"].iloc[:-1], test["y"].iloc[:-1], const[:-1], results_dir=tmp)
    except ValueError as e:
        print("Missing days refused:", e)

      date    y_true    y_pred
2015-01-02 -3.477858 -4.027267
2015-01-05 -3.645751 -4.027267
2015-01-06 -3.708380 -4.027267
Missing days refused: predictions must cover exactly the 717 test dates (2015-01-02 to 2017-11-03), in order
